### Building a Rag system with Langchain and ChromaDB
#### INTRODUCTION
Retrieval augmented generation is a powerful technique that combines the capabilities of large language model with external knowledge retrieval. This notebook builds a rag system using :

-Langchain : A framework for developing applications powered by language models

-ChromaDB: an opensource vector database for storing and retrieving embeddings

-groq: for embeddings and language model

In [2]:
from dotenv import load_dotenv
load_dotenv()

True

In [3]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import TextLoader, DirectoryLoader
from langchain_core.documents import Document
from langchain_huggingface import HuggingFaceEmbeddings

# vector stores
from langchain_community.vectorstores import Chroma

/Users/mayanksharma/Desktop/Projects/Rag/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
sample_docs=['''
Machine Learning

Machine Learning is a branch of artificial intelligence that allows computers to learn patterns from data and make predictions without being explicitly programmed.
''',
'''Deep Learning

Deep Learning is a subset of machine learning that uses neural networks with multiple layers to process complex data such as images, speech, and text.
''',
'''Natural Language Processing

Natural Language Processing, or NLP, enables computers to understand, interpret, and generate human language for tasks like translation, chatbots, and sentiment analysis.
''']

sample_docs

['\nMachine Learning\n\nMachine Learning is a branch of artificial intelligence that allows computers to learn patterns from data and make predictions without being explicitly programmed.\n',
 'Deep Learning\n\nDeep Learning is a subset of machine learning that uses neural networks with multiple layers to process complex data such as images, speech, and text.\n',
 'Natural Language Processing\n\nNatural Language Processing, or NLP, enables computers to understand, interpret, and generate human language for tasks like translation, chatbots, and sentiment analysis.\n']

In [5]:
import tempfile
temp_dir=tempfile.mkdtemp()

for i,doc in enumerate(sample_docs):
    with open(f"doc_{i}.txt","w") as f:
        temp_dir = tempfile.mkdtemp()

        for i, doc in enumerate(sample_docs):
            with open(f"doc_{i}.txt", "w", encoding="utf-8") as f:
                f.write(doc)

In [6]:
from langchain_community.document_loaders import DirectoryLoader,TextLoader
loader=DirectoryLoader(
    "data",
    glob="*.txt",
    loader_cls=TextLoader,
    loader_kwargs={'encoding':'utf-8'}
)


In [7]:
#document spliting 
text_splitter=RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50,
    length_function=len,
    separators=[" "]
)
docs=loader.load()
chunks=text_splitter.split_documents(docs)
print(f'created {len(chunks)} chunks')

created 3 chunks


In [8]:
sample_text='Machine Learning is facinating'
embeddings=HuggingFaceEmbeddings(model_name='sentence-transformers/all-MiniLM-L6-v2')

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 7103.50it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [9]:
vector=embeddings.embed_query(sample_text)
vector

[-0.011421228758990765,
 -0.09401243925094604,
 0.07909875363111496,
 0.010052294470369816,
 0.03052470088005066,
 -0.07159080356359482,
 0.025884568691253662,
 -0.04600033909082413,
 -0.010964811779558659,
 -0.0049161906354129314,
 -0.08440322428941727,
 0.017160436138510704,
 0.0356481559574604,
 -0.03660527616739273,
 -0.16236743330955505,
 -0.05686252936720848,
 -0.007439390756189823,
 -0.04244780167937279,
 -0.052435532212257385,
 -0.06360574811697006,
 -0.05709031969308853,
 -0.010202301666140556,
 -0.03527824953198433,
 0.01609625667333603,
 0.07030285149812698,
 -0.014181118458509445,
 0.021265968680381775,
 0.0102001391351223,
 0.009754017926752567,
 -0.056863315403461456,
 0.03896918520331383,
 0.038046181201934814,
 -0.0020143049769103527,
 -0.011471720412373543,
 -0.10078305751085281,
 -0.013842426240444183,
 0.042432140558958054,
 0.06592868268489838,
 0.03540670871734619,
 0.005910211708396673,
 0.007874393835663795,
 -0.06986445933580399,
 -0.008599887602031231,
 0.03838

### Intialize the chromadb vector store and stores the chunks in vector representation

In [10]:
### create a chromadb vector store 
persist_directory='./_new_chroma_db'

## initialize chromadb with openai embeddings 
vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=HuggingFaceEmbeddings(),
    persist_directory=persist_directory,
    collection_name='rag_collection'
)

print(f'vector store created with {vectorstore._collection.count()} vectors')
print(f'persisted to : {persist_directory}')

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 6218.00it/s]
MPNetModel LOAD REPORT from: sentence-transformers/all-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


vector store created with 9 vectors
persisted to : ./_new_chroma_db


### test similarity search

In [11]:
query='what are the types of machine learning'

similarity_docs=vectorstore.similarity_search(query, k=3)
similarity_docs

[Document(metadata={'source': 'data/doc_0.txt'}, page_content='Machine Learning\n\nMachine Learning is a branch of artificial intelligence that allows computers to learn patterns from data and make predictions without being explicitly programmed.'),
 Document(metadata={'source': 'data/doc_0.txt'}, page_content='Machine Learning\n\nMachine Learning is a branch of artificial intelligence that allows computers to learn patterns from data and make predictions without being explicitly programmed.'),
 Document(metadata={'source': 'data/doc_0.txt'}, page_content='Machine Learning\n\nMachine Learning is a branch of artificial intelligence that allows computers to learn patterns from data and make predictions without being explicitly programmed.')]

In [12]:
query='what is deep learning'

similarity_docs=vectorstore.similarity_search(query, k=3)
similarity_docs

[Document(metadata={'source': 'data/doc_1.txt'}, page_content='Deep Learning\n\nDeep Learning is a subset of machine learning that uses neural networks with multiple layers to process complex data such as images, speech, and text.'),
 Document(metadata={'source': 'data/doc_1.txt'}, page_content='Deep Learning\n\nDeep Learning is a subset of machine learning that uses neural networks with multiple layers to process complex data such as images, speech, and text.'),
 Document(metadata={'source': 'data/doc_1.txt'}, page_content='Deep Learning\n\nDeep Learning is a subset of machine learning that uses neural networks with multiple layers to process complex data such as images, speech, and text.')]

In [13]:
### advances similarity search with scores 
result_scores=vectorstore.similarity_search_with_score(query,k=3)
result_scores

[(Document(metadata={'source': 'data/doc_1.txt'}, page_content='Deep Learning\n\nDeep Learning is a subset of machine learning that uses neural networks with multiple layers to process complex data such as images, speech, and text.'),
  0.40769246220588684),
 (Document(metadata={'source': 'data/doc_1.txt'}, page_content='Deep Learning\n\nDeep Learning is a subset of machine learning that uses neural networks with multiple layers to process complex data such as images, speech, and text.'),
  0.40769246220588684),
 (Document(metadata={'source': 'data/doc_1.txt'}, page_content='Deep Learning\n\nDeep Learning is a subset of machine learning that uses neural networks with multiple layers to process complex data such as images, speech, and text.'),
  0.40769246220588684)]

In [14]:
# Reload environment variables to ensure GROQ_API_KEY is loaded
from dotenv import load_dotenv
import os
load_dotenv(override=True)

# Using Groq free API
from langchain_groq import ChatGroq

llm = ChatGroq(
    model="llama-3.3-70b-versatile",  
    temperature=0
)

test_response=llm.invoke("what is llm")
test_response

AIMessage(content='LLM stands for Large Language Model. It refers to a type of artificial intelligence (AI) designed to process and understand human language at a large scale. LLMs are trained on vast amounts of text data, which enables them to learn patterns, relationships, and structures of language.\n\nLarge Language Models are typically based on deep learning architectures, such as transformer models, and are trained using self-supervised learning techniques. This means that they learn to predict the next word in a sentence or to fill in missing words, without being explicitly told what to do.\n\nLLMs have many applications, including:\n\n1. **Language translation**: LLMs can be used to translate text from one language to another.\n2. **Text summarization**: LLMs can summarize long pieces of text into shorter, more digestible versions.\n3. **Chatbots and conversational AI**: LLMs can be used to power chatbots and other conversational interfaces.\n4. **Language generation**: LLMs ca

In [15]:
#easier initiating chat model for lanchain
from langchain.chat_models.base import init_chat_model

llm=init_chat_model('groq:llama-3.3-70b-versatile')
test_response=llm.invoke("what is llm")
test_response

AIMessage(content='LLM stands for Large Language Model. It refers to a type of artificial intelligence (AI) designed to process and understand human language at a large scale. LLMs are trained on vast amounts of text data, which enables them to learn patterns, relationships, and structures within language.\n\nCharacteristics of LLMs:\n\n1. **Scalability**: LLMs are trained on massive datasets, often consisting of billions of words or more.\n2. **Language understanding**: LLMs aim to comprehend the nuances of language, including grammar, syntax, semantics, and context.\n3. **Generative capabilities**: LLMs can generate text, summarize content, translate languages, and even create new text based on a given prompt.\n4. **Deep learning architecture**: LLMs typically employ deep learning architectures, such as transformers, to process and analyze language.\n\nApplications of LLMs:\n\n1. **Language translation**: LLMs can translate languages with high accuracy, enabling global communication 

In [16]:
from langchain_classic.chains import create_retrieval_chain
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate

In [17]:
## convert vector stor to retriver 
retriever=vectorstore.as_retriever(
    search_kwarg={'k':3}
)
retriever

VectorStoreRetriever(tags=['Chroma', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.chroma.Chroma object at 0x30cc8cc20>, search_kwargs={})

In [18]:
# create a prompt template 
from langchain_core.prompts import ChatPromptTemplate

system_promt="""you are a assistant for question answering tasks.
use the following pieces of retrived context to ansert the question.
if you don't know the answer,just say that you don't know.
use three sentences maximum and keeo the answer consise.

context:{context}"""

prompt=ChatPromptTemplate.from_messages([
    ("system",system_promt),
    ("human","question:{question}")
])

In [19]:
### create a document chain 
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
document_chain=create_stuff_documents_chain(llm,prompt)
document_chain


RunnableBinding(bound=RunnableBinding(bound=RunnableAssign(mapper={
  context: RunnableLambda(format_docs)
}), kwargs={}, config={'run_name': 'format_inputs'}, config_factories=[])
| ChatPromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context'], input_types={}, partial_variables={}, template="you are a assistant for question answering tasks.\nuse the following pieces of retrived context to ansert the question.\nif you don't know the answer,just say that you don't know.\nuse three sentences maximum and keeo the answer consise.\n\ncontext:{context}"), additional_kwargs={}), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['question'], input_types={}, partial_variables={}, template='question:{question}'), additional_kwargs={})])
| ChatGroq(profile={'max_input_tokens': 131072, 'max_output_tokens': 32768, 'image_inputs': False, 'audio_inputs': False, '

In [20]:
## FINAL RAG CHAIN 
rag_chain=create_retrieval_chain(retriever,document_chain)
rag_chain

RunnableBinding(bound=RunnableAssign(mapper={
  context: RunnableBinding(bound=RunnableLambda(lambda x: x['input'])
           | VectorStoreRetriever(tags=['Chroma', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.chroma.Chroma object at 0x30cc8cc20>, search_kwargs={}), kwargs={}, config={'run_name': 'retrieve_documents'}, config_factories=[])
})
| RunnableAssign(mapper={
    answer: RunnableBinding(bound=RunnableBinding(bound=RunnableAssign(mapper={
              context: RunnableLambda(format_docs)
            }), kwargs={}, config={'run_name': 'format_inputs'}, config_factories=[])
            | ChatPromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context'], input_types={}, partial_variables={}, template="you are a assistant for question answering tasks.\nuse the following pieces of retrived context to ansert the question.\nif you don't k

In [24]:
response = rag_chain.invoke({
    "input": "what is Deep Learning",
    "question": "what is Deep Learning"
})
response
response

{'input': 'what is Deep Learning',
 'question': 'what is Deep Learning',
 'context': [Document(metadata={'source': 'data/doc_1.txt'}, page_content='Deep Learning\n\nDeep Learning is a subset of machine learning that uses neural networks with multiple layers to process complex data such as images, speech, and text.'),
  Document(metadata={'source': 'data/doc_1.txt'}, page_content='Deep Learning\n\nDeep Learning is a subset of machine learning that uses neural networks with multiple layers to process complex data such as images, speech, and text.'),
  Document(metadata={'source': 'data/doc_1.txt'}, page_content='Deep Learning\n\nDeep Learning is a subset of machine learning that uses neural networks with multiple layers to process complex data such as images, speech, and text.'),
  Document(metadata={'source': 'data/doc_0.txt'}, page_content='Machine Learning\n\nMachine Learning is a branch of artificial intelligence that allows computers to learn patterns from data and make predictions 